In [11]:
from transformers import PretrainedConfig
from transformers.models.qwen3.configuration_qwen3 import Qwen3Config
from typing import Optional, Union
import torch
import torch.nn as nn
import os
from torch.export import Dim

In [2]:
class OmniVoiceConfig(PretrainedConfig):
    model_type = "omnivoice"
    sub_configs = {"llm_config": Qwen3Config}

    def __init__(
        self,
        audio_vocab_size: int = 1025,
        audio_mask_id: int = 1024,
        num_audio_codebook: int = 8,
        audio_codebook_weights: Optional[list[float]] = None,
        llm_config: Optional[Union[dict, Qwen3Config]] = None,
        **kwargs,
    ):
        llm_config = Qwen3Config(**llm_config)
        self.llm_config = llm_config
        
        super().__init__(**kwargs)
        self.audio_vocab_size = audio_vocab_size
        self.audio_mask_id = audio_mask_id
        self.num_audio_codebook = num_audio_codebook
        self.audio_codebook_weights = audio_codebook_weights

In [3]:
def _resolve_model_path(name_or_path: str) -> str:
    if os.path.isdir(name_or_path):
        return name_or_path
    from huggingface_hub import snapshot_download

    return snapshot_download(name_or_path)

In [4]:
model_path = _resolve_model_path("k2-fsa/OmniVoice")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

In [5]:
config = OmniVoiceConfig.from_pretrained(model_path)

In [6]:
class AudioEmbeddings(nn.Module):
    def __init__(self, embedding: nn.Embedding):
        super().__init__()
        self.embedding = embedding

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        emb = self.embedding(x)        
        return emb.sum(dim=1)            


class AudioHeads(nn.Module):
    def __init__(self, linear: nn.Linear):
        super().__init__()
        self.linear = linear

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear(x)

In [7]:
raw_emb = nn.Embedding(
    config.num_audio_codebook * config.audio_vocab_size,   
    config.llm_config.hidden_size,                         
)
raw_emb.load_state_dict(
    torch.load(
        "./omnivoice_audio_embeddings/model.pt",
        map_location="cpu",
        weights_only=True,
    )
)
audio_embeddings = AudioEmbeddings(raw_emb)
audio_embeddings.eval()

raw_heads = nn.Linear(
    config.llm_config.hidden_size,                         
    config.num_audio_codebook * config.audio_vocab_size,   
    bias=False,
)
raw_heads.load_state_dict(
    torch.load(
        "./omnivoice_audio_heads/model.pt",
        map_location="cpu",
        weights_only=True,
    )
)
audio_heads = AudioHeads(raw_heads)
audio_heads.eval()

AudioHeads(
  (linear): Linear(in_features=1024, out_features=8200, bias=False)
)

In [8]:
B = 2
NUM_CB = 8          
T = 220             
HIDDEN = 1024      
OUT_DIM = 8200      

dummy_ids = torch.randint(
    0,
    config.num_audio_codebook * config.audio_vocab_size,
    (B, NUM_CB, T),
    dtype=torch.long,
)

dummy_hidden = torch.randn(B, T, HIDDEN, dtype=torch.float32)

In [9]:
AUDIO_EMBEDDINGS_PATH = "./onnx_omnivoice_audio_embeddings"
AUDIO_HEADS_PATH = "./onnx_omnivoice_audio_heads"

In [10]:
os.makedirs(AUDIO_EMBEDDINGS_PATH, exist_ok=True)
os.makedirs(AUDIO_HEADS_PATH, exist_ok=True)

In [13]:
torch.onnx.export(
    audio_embeddings,
    (dummy_ids,),
    os.path.join(AUDIO_EMBEDDINGS_PATH, "model.onnx"),
    input_names=["input_ids"],
    output_names=["audio_embeds"],
    dynamic_axes={
        "input_ids": {0: "batch", 2: "seq_len"},   
        "audio_embeds": {0: "batch", 1: "seq_len"},
    },
    opset_version=17,
    do_constant_folding=True,
    external_data=False,
)

/tmp/ipykernel_53585/559823211.py:1: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W1101 21:04:53.788000 53585 site-packages/torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W1101 21:04:56.389000 53585 site-packages/torch/onnx/_internal/exporter/_registration.py:107] torchvision is not installed. Skipping torchvision::nms
W1101 21:04:56.390000 53585 site-packages/torch/onnx/_intern

[torch.onnx] Obtain model graph for `AudioEmbeddings([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `AudioEmbeddings([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 17},
            producer_name='pytorch',
            producer_version='2.14.1+cu130',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"input_ids"<INT64,[batch,8,seq_len]>
            ),
            outputs=(
                %"audio_embeds"<FLOAT,[batch,seq_len,1024]>
            ),
            initializers=(
                %"embedding.weight"<FLOAT,[8200,1024]>{TorchTensor(...)},
                %"val_0"<INT64,[1]>{TensorProtoTensor<INT64,[1]>(array([1]), name='val_0')}
            ),
        ) {
            0 |  # node_embedding
                 %"embedding"<FLOAT,[batch,8,seq_len,1024]> ⬅️ ::Gather(%"embedding.weight"{...}, %"input_ids") {axis=0}
            1 |  # node_sum_1
                 %"audio_embeds"<FLOAT,[batch,seq_len,1024]> ⬅️ ::ReduceSum(%"embedding", %"val_0"{[1]}) {noop_with_em

In [14]:
torch.onnx.export(
    audio_heads,
    (dummy_hidden,),
    os.path.join(AUDIO_HEADS_PATH, "model.onnx"),
    input_names=["hidden_states"],
    output_names=["logits"],
    dynamic_axes={
        "hidden_states": {0: "batch", 1: "seq_len"},  
        "logits": {0: "batch", 1: "seq_len"},
    },
    opset_version=17,
    do_constant_folding=True,
    external_data=False,
)

/tmp/ipykernel_53585/184019736.py:1: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W1101 21:05:32.307000 53585 site-packages/torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W1101 21:05:32.891000 53585 site-packages/torch/onnx/_internal/exporter/_registration.py:107] torchvision is not installed. Skipping torchvision::nms
W1101 21:05:32.892000 53585 site-packages/torch/onnx/_intern

[torch.onnx] Obtain model graph for `AudioHeads([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `AudioHeads([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 17},
            producer_name='pytorch',
            producer_version='2.14.1+cu130',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"hidden_states"<FLOAT,[batch,seq_len,1024]>
            ),
            outputs=(
                %"logits"<FLOAT,[batch,seq_len,8200]>
            ),
            initializers=(
                %"val_0"<FLOAT,[1024,8200]>{Tensor(...)}
            ),
        ) {
            0 |  # node_linear
                 %"logits"<FLOAT,[batch,seq_len,8200]> ⬅️ ::MatMul(%"hidden_states", %"val_0"{...})
            return %"logits"<FLOAT,[batch,seq_len,8200]>
        }


    ,
    exported_program=
        ExportedProgram:
            class GraphModule(torch.nn.Module):
                def forward(self, p_linear_weight: "f32[8200, 1024]", x: "f32[s77, s27, 1024]"):
             